# 15b - Valutazione MaxViT su sintetiche from-scratch SD-VAE

Questo notebook allena due classificatori MaxViT-Tiny-512 usando solo le immagini sintetiche prodotte dal diffusore from scratch con VAE Stable Diffusion (`data/synthetic/fromscratch_new`).

Serve a decidere se per questa sorgente sintetica conviene un fine-tuning parziale dell'ultimo stage o lo sblocco completo del backbone.


In [ ]:
# === Bootstrap unificato notebooks/ ===
# Funziona dalla root del progetto e da ogni sottocartella della struttura notebooks/.
import sys as _sys
from pathlib import Path as _Path


def _find_mammo_root():
    for _candidate in [_Path.cwd().resolve(), *_Path.cwd().resolve().parents]:
        if _candidate.name == "MammoDiffusion":
            return _candidate
        if (_candidate / "data").is_dir() and (_candidate / "notebooks").is_dir():
            return _candidate
    raise FileNotFoundError("Root MammoDiffusion non trovata da " + str(_Path.cwd()))


PROJECT_ROOT = _find_mammo_root()
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
UTILITY_DIR = NOTEBOOKS_DIR / "utility"
for _path in (str(UTILITY_DIR), str(NOTEBOOKS_DIR)):
    if _path not in _sys.path:
        _sys.path.insert(0, _path)

BASE = PROJECT_ROOT
BASE_DIR = PROJECT_ROOT
BASE_PATH = str(PROJECT_ROOT) + "/"
# === Fine bootstrap unificato ===

import os
import sys
from pathlib import Path

# [notebooks bootstrap] Il blocco Colab/local originale e' stato
# neutralizzato: BASE_PATH e BASE sono gia' definiti dal bootstrap
# unificato in cima al notebook.
print("Percorso base:", BASE)

Ambiente locale rilevato.
Percorso base: /mnt/MammoDiffusion/MammoDiffusion


In [2]:
import json
import zipfile
import re
from datetime import date
from pathlib import Path

import gdown
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)

sys.path.insert(0, os.getcwd())
from maxvit_utils import (
    build_maxvit_model,
    resolve_normalization,
    make_dataloader,
    freeze_all,
    unfreeze_head,
    unfreeze_stages_from,
    unfreeze_all,
    count_trainable_params,
    BinaryFocalLoss,
    compute_pos_weight,
    EarlyStopping,
    ModelCheckpoint,
    CSVLogger,
    fit,
    predict_probs,
    optimal_threshold_youden,
    bootstrap_balanced,
)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", DEVICE)


Device: cuda


In [3]:
PROCESSED_DRIVE_ID = "1qQral_BIBlMl0QN3PllJukdYTOmNGWr3"
PROCESSED_ZIP = BASE / 'data' / 'processed' / 'processed.zip'

TRAIN_CSV_PATH = BASE / 'data' / 'processed' / 'metadata' / 'train.csv'
VAL_CSV_PATH = BASE / 'data' / 'processed' / 'metadata' / 'val.csv'
TEST_CSV_PATH = BASE / 'data' / 'processed' / 'metadata' / 'test.csv'

def processed_data_ready():
    required = [
        TRAIN_CSV_PATH,
        VAL_CSV_PATH,
        TEST_CSV_PATH,
        BASE / 'data' / 'processed' / 'train' / '0',
        BASE / 'data' / 'processed' / 'train' / '1',
        BASE / 'data' / 'processed' / 'val' / '0',
        BASE / 'data' / 'processed' / 'val' / '1',
    ]
    return all(p.exists() for p in required)

if not processed_data_ready():
    print("Dataset preprocessato non trovato, scarico da Google Drive...")
    PROCESSED_ZIP.parent.mkdir(parents=True, exist_ok=True)
    gdown.download(id=PROCESSED_DRIVE_ID, output=str(PROCESSED_ZIP), quiet=False)
    with zipfile.ZipFile(PROCESSED_ZIP, 'r') as z:
        z.extractall(BASE / 'data' / 'processed')
    PROCESSED_ZIP.unlink(missing_ok=True)
    print("Dataset estratto in:", BASE / 'data' / 'processed')
else:
    print("Dataset preprocessato gia presente.")


Dataset preprocessato gia presente.


In [4]:
SYNTH_ROOT = BASE / 'data' / 'synthetic' / 'fromscratch_new'
SYNTHETIC_SOURCE_NAME = 'fromscratch_sdvae'
OUTPUT_CSV = BASE / 'data' / 'synthetic' / 'metadata' / 'train_synthetic_fromscratch_sdvae.csv'

RESULTS_DIR = BASE / 'results' / 'classifiers/maxvit512/02w_validation_fromscratch_partial_vs_full'
FIGURES_DIR = RESULTS_DIR / 'figures'
TABLES_DIR = RESULTS_DIR / 'tables'
PREDICTIONS_DIR = RESULTS_DIR / 'predictions'
for directory in [OUTPUT_CSV.parent, FIGURES_DIR, TABLES_DIR, PREDICTIONS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

VARIANT_SPECS = [
    {
        'name': 'FromScratch_SD_VAE_Partial',
        'fine_tune_mode': 'partial',
        'experiment_dir': BASE / 'experiments' / 'classifiers/maxvit512/02g_fromscratch_synthetic_only_partial',
        'checkpoint_prefix': 'fromscratch_sdvae_synth_maxvit512_partial',
    },
    {
        'name': 'FromScratch_SD_VAE_Full',
        'fine_tune_mode': 'full',
        'experiment_dir': BASE / 'experiments' / 'classifiers/maxvit512/02h_fromscratch_synthetic_only_full',
        'checkpoint_prefix': 'fromscratch_sdvae_synth_maxvit512_full',
    },
]

print("Sintetiche from-scratch SD-VAE attese in:", SYNTH_ROOT)
print("Output confronto:", RESULTS_DIR)


Sintetiche from-scratch SD-VAE attese in: /mnt/MammoDiffusion/MammoDiffusion/data/synthetic/fromscratch_new
Output confronto: /mnt/MammoDiffusion/MammoDiffusion/results/15b_val_classificatori_fromscratch_sdvae_maxvit512_allVSpart


In [5]:
IMAGE_EXTENSIONS = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}

def image_paths(directory):
    directory = Path(directory)
    return sorted(
        p for p in directory.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS and not p.name.startswith('.')
    )

def load_real_split(split):
    rows = []
    split_dir = BASE / 'data' / 'processed' / split
    for label in [0, 1]:
        for path in image_paths(split_dir / str(label)):
            rows.append({
                'processed_path': str(path),
                'cancer': label,
                'source': 'real',
                'source_detail': f'real_{split}',
            })
    df = pd.DataFrame(rows)
    if df.empty:
        raise FileNotFoundError(f"Nessuna immagine reale trovata in {split_dir}")
    return df

def load_synthetic_both(root, source_name):
    root = Path(root)
    rows = []
    for folder, label in [('negative', 0), ('positive', 1)]:
        folder_path = root / folder
        paths = image_paths(folder_path) if folder_path.is_dir() else []
        if not paths:
            raise FileNotFoundError(
                f"Nessuna immagine sintetica {folder} in {folder_path}. "
                "Esegui prima il notebook generativo corrispondente."
            )
        for path in paths:
            rows.append({
                'processed_path': str(path),
                'cancer': label,
                'source': 'synthetic',
                'source_detail': source_name,
            })
    return pd.DataFrame(rows)

_LABEL_RE = re.compile(r"_label([01])")

def load_augmented_positive():
    aug_dir = BASE / 'data' / 'real_augmented'
    rows = []
    if not aug_dir.is_dir():
        raise FileNotFoundError(
            f"Cartella {aug_dir} mancante. Esegui prima 02_Data_Augmentation_Trad.ipynb."
        )
    for path in image_paths(aug_dir):
        match = _LABEL_RE.search(path.name)
        if match is None:
            continue
        rows.append({
            'processed_path': str(path),
            'cancer': int(match.group(1)),
            'source': 'augmented',
            'source_detail': 'traditional_positive_augmentation',
        })
    df = pd.DataFrame(rows)
    if df.empty:
        raise FileNotFoundError(f"Nessuna immagine augmentata riconosciuta in {aug_dir}")
    return df

def print_counts(name, df):
    labels = df['cancer'].astype(int)
    print(
        f"{name:18s} tot={len(df):5d} | sano(0)={(labels == 0).sum():5d} | "
        f"malato(1)={(labels == 1).sum():5d}"
    )

def source_table(df):
    table = pd.crosstab(df['source_detail'], df['cancer'])
    table = table.rename(columns={0: 'sano_0', 1: 'malato_1'})
    table['totale'] = table.sum(axis=1)
    return table


In [ ]:
df_synth = load_synthetic_both(SYNTH_ROOT, SYNTHETIC_SOURCE_NAME)
df_val = load_real_split('val')
df_test = load_real_split('test')

df_synth.to_csv(OUTPUT_CSV, index=False)
print("CSV synthetic-only salvato in:", OUTPUT_CSV)
print_counts("TRAIN synthetic", df_synth)
print_counts("VAL real", df_val)
print_counts("TEST real", df_test)
display(source_table(df_synth))


In [ ]:
IMG_SIZE = 512
BATCH_SIZE = 8
SEED = 42
IMAGENET_MEAN = (0.5, 0.5, 0.5)
IMAGENET_STD = (0.5, 0.5, 0.5)

torch.manual_seed(SEED)
np.random.seed(SEED)

def make_loaders(df_train, df_val, df_test=None):
    train_loader = make_dataloader(
        df_train, 'processed_path', 'cancer', IMAGENET_MEAN, IMAGENET_STD, IMG_SIZE,
        batch_size=BATCH_SIZE, shuffle=True, augment=False, seed=SEED,
    )
    val_loader = make_dataloader(
        df_val, 'processed_path', 'cancer', IMAGENET_MEAN, IMAGENET_STD, IMG_SIZE,
        batch_size=BATCH_SIZE, shuffle=False,
    )
    test_loader = None
    if df_test is not None:
        test_loader = make_dataloader(
            df_test, 'processed_path', 'cancer', IMAGENET_MEAN, IMAGENET_STD, IMG_SIZE,
            batch_size=BATCH_SIZE, shuffle=False,
        )
    return train_loader, val_loader, test_loader

def apply_fine_tune_mode(model, fine_tune_mode):
    if fine_tune_mode == 'partial':
        unfreeze_stages_from(model, start_stage=3)
        return "stage 3/4 (ultimo stage)"
    if fine_tune_mode == 'full':
        unfreeze_all(model)
        return "tutti gli stage"
    raise ValueError(f"fine_tune_mode non supportato: {fine_tune_mode}")

def plot_training_history(history, title_suffix, save_path):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(history.history['loss'], label='Train Loss', linewidth=2)
    ax1.plot(history.history['val_loss'], label='Val Loss', linewidth=2)
    ax1.set_title(f'Loss {title_suffix}')
    ax1.set_xlabel('Epoca')
    ax1.set_ylabel('Loss')
    ax1.legend()
    ax1.grid(True, linestyle='--', alpha=0.6)

    ax2.plot(history.history['auc'], label='Train AUC', linewidth=2)
    ax2.plot(history.history['val_auc'], label='Val AUC', linewidth=2)
    ax2.set_title(f'AUC {title_suffix}')
    ax2.set_xlabel('Epoca')
    ax2.set_ylabel('AUC')
    ax2.legend()
    ax2.grid(True, linestyle='--', alpha=0.6)

    fig.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print("Grafico salvato in:", save_path)

def metrics_from_probs(y_true, y_prob, threshold, split, experiment_name, config_name):
    y_pred = (y_prob >= threshold).astype(int)
    report = classification_report(y_true, y_pred, target_names=['Sano', 'Malato'])
    return {
        'arch': 'MaxViT-Tiny-512',
        'config': config_name,
        'split': split,
        'experiment_name': experiment_name,
        'threshold_youden_from_val': round(float(threshold), 4),
        'roc_auc': round(float(roc_auc_score(y_true, y_prob)), 4),
        'accuracy': round(float((y_pred == y_true).mean()), 4),
        'precision_malato': round(float(precision_score(y_true, y_pred, pos_label=1, zero_division=0)), 4),
        'recall_malato': round(float(recall_score(y_true, y_pred, pos_label=1, zero_division=0)), 4),
        'f1_malato': round(float(f1_score(y_true, y_pred, pos_label=1, zero_division=0)), 4),
        'classification_report': report,
    }, y_pred

def save_confusion_roc(y_true, y_prob, y_pred, title, save_path):
    cm = confusion_matrix(y_true, y_pred)
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc = roc_auc_score(y_true, y_prob)
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].imshow(cm, cmap='Blues')
    ax[0].set_xticks([0, 1]); ax[0].set_yticks([0, 1])
    ax[0].set_xticklabels(['Sano', 'Malato'])
    ax[0].set_yticklabels(['Sano', 'Malato'])
    ax[0].set_xlabel('Predetto'); ax[0].set_ylabel('Reale')
    ax[0].set_title('Confusion Matrix')
    for i in range(2):
        for j in range(2):
            color = 'white' if cm[i, j] > cm.max() / 2 else 'black'
            ax[0].text(j, i, cm[i, j], ha='center', va='center',
                       fontsize=14, fontweight='bold', color=color)
    ax[1].plot(fpr, tpr, lw=2, label=f'AUC = {auc:.4f}')
    ax[1].plot([0, 1], [0, 1], '--', color='gray')
    ax[1].set_xlabel('FPR'); ax[1].set_ylabel('TPR')
    ax[1].set_title('ROC')
    ax[1].legend()
    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print("Figura salvata in:", save_path)

def train_maxvit_experiment(df_train, df_val, output_dir, checkpoint_prefix, config_name, fine_tune_mode):
    output_dir = Path(output_dir)
    figures_dir = output_dir / 'figures'
    output_dir.mkdir(parents=True, exist_ok=True)
    figures_dir.mkdir(parents=True, exist_ok=True)

    train_loader, val_loader, _ = make_loaders(df_train, df_val)

    model = build_maxvit_model(num_classes=1, pretrained=True)
    model.to(DEVICE)
    resolved_mean, resolved_std, resolved_size = resolve_normalization(model)
    assert resolved_size == IMG_SIZE, f"IMG_SIZE atteso {resolved_size}, trovato {IMG_SIZE}"
    print(f"Normalizzazione modello: mean={resolved_mean} std={resolved_std} input_size={resolved_size}")

    experiment_config = {
        'experiment_name': output_dir.name,
        'date': date.today().isoformat(),
        'config': config_name,
        'fine_tune_mode': fine_tune_mode,
        'backbone': {
            'name': 'maxvit_tiny_tf_512.in1k',
            'library': 'timm',
            'weights': 'in1k',
            'input_shape': [IMG_SIZE, IMG_SIZE, 3],
        },
        'data': {
            'batch_size': BATCH_SIZE,
            'seed': SEED,
            'n_train': int(len(df_train)),
            'n_val': int(len(df_val)),
            'source_counts': source_table(df_train).to_dict(orient='index'),
        },
        'phase1_head_training': {
            'optimizer': 'Adam',
            'learning_rate': 1e-3,
            'loss': 'BCEWithLogitsLoss(pos_weight=balanced)',
            'max_epochs': 25,
            'early_stopping_patience': 7,
            'backbone_trainable': False,
        },
        'phase2_fine_tuning': {
            'optimizer': 'Adam',
            'learning_rate': 1e-5,
            'loss': 'BinaryFocalLoss(alpha=0.75, gamma=2.0)',
            'max_epochs': 30,
            'fine_tune_from': fine_tune_mode,
            'early_stopping_patience': 5,
            'early_stopping_min_delta': 1e-3,
        },
    }
    with open(output_dir / 'experiment_config.json', 'w') as f:
        json.dump(experiment_config, f, indent=2, ensure_ascii=False)

    freeze_all(model)
    unfreeze_head(model)
    trainable, total = count_trainable_params(model)
    print(f"Parametri trainable (Fase 1): {trainable:,} / {total:,}")

    pos_weight = compute_pos_weight(df_train['cancer'].values).to(DEVICE)
    criterion_fase1 = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer_fase1 = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=1e-3)
    history_phase1 = fit(
        model, train_loader, val_loader, optimizer_fase1, criterion_fase1,
        epochs=25, device=DEVICE,
        early_stopping=EarlyStopping(patience=7, mode='max', restore_best_weights=True),
        checkpoint=ModelCheckpoint(str(output_dir / f'{checkpoint_prefix}_fase1_best.pt'), mode='max'),
        csv_logger=CSVLogger(str(output_dir / 'training_log_fase1.csv')),
    )

    fine_tune_desc = apply_fine_tune_mode(model, fine_tune_mode)
    trainable, total = count_trainable_params(model)
    print(f"Parametri trainable (Fase 2 - {fine_tune_desc}): {trainable:,} / {total:,}")

    criterion_fase2 = BinaryFocalLoss(alpha=0.75, gamma=2.0)
    optimizer_fase2 = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=1e-5)
    lr_scheduler_fase2 = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer_fase2, mode='max', factor=0.5, patience=3, min_lr=1e-7
    )
    final_ckpt = output_dir / f'{checkpoint_prefix}_final_best.pt'
    history_phase2 = fit(
        model, train_loader, val_loader, optimizer_fase2, criterion_fase2,
        epochs=30, device=DEVICE,
        early_stopping=EarlyStopping(patience=5, min_delta=1e-3, mode='max', restore_best_weights=True),
        checkpoint=ModelCheckpoint(str(final_ckpt), mode='max'),
        csv_logger=CSVLogger(str(output_dir / 'training_log_fase2.csv')),
        lr_scheduler=lr_scheduler_fase2,
    )

    plot_training_history(history_phase1, '(Fase 1)', figures_dir / 'training_history_fase1.png')
    plot_training_history(history_phase2, f'(Fase 2 - {fine_tune_mode})', figures_dir / 'training_history_fase2.png')

    model.load_state_dict(torch.load(final_ckpt, map_location=DEVICE))
    y_true, y_prob = predict_probs(model, val_loader, DEVICE)
    threshold = optimal_threshold_youden(y_true, y_prob)
    metrics, y_pred = metrics_from_probs(y_true, y_prob, threshold, 'validation', output_dir.name, config_name)
    metrics['optimal_threshold_youden'] = metrics['threshold_youden_from_val']
    metrics['fine_tune_mode'] = fine_tune_mode
    metrics['checkpoint'] = str(final_ckpt)
    with open(output_dir / 'val_metrics.json', 'w') as f:
        json.dump(metrics, f, indent=2, ensure_ascii=False)
    save_confusion_roc(
        y_true, y_prob, y_pred,
        f'{config_name} - validation - {fine_tune_mode}',
        figures_dir / 'val_confusion_roc.png',
    )
    print(f"Training completato. Miglior modello: {final_ckpt}")
    print(json.dumps({k: metrics[k] for k in ['roc_auc', 'f1_malato', 'recall_malato', 'precision_malato']}, indent=2))
    return {
        'model': model,
        'metrics': metrics,
        'y_true': y_true,
        'y_prob': y_prob,
        'threshold': threshold,
        'checkpoint': final_ckpt,
        'output_dir': output_dir,
    }


In [ ]:
variant_outputs = {}
comparison_rows = []

for spec in VARIANT_SPECS:
    print("\n" + "=" * 80)
    print("Avvio variante:", spec['name'])
    result = train_maxvit_experiment(
        df_train=df_synth,
        df_val=df_val,
        output_dir=spec['experiment_dir'],
        checkpoint_prefix=spec['checkpoint_prefix'],
        config_name=spec['name'],
        fine_tune_mode=spec['fine_tune_mode'],
    )
    variant_outputs[spec['name']] = result
    row = {
        'name': spec['name'],
        'fine_tune_mode': spec['fine_tune_mode'],
        'experiment_dir': str(spec['experiment_dir']),
        'checkpoint': str(result['checkpoint']),
        **{k: v for k, v in result['metrics'].items() if k not in ['classification_report']},
    }
    comparison_rows.append(row)

    del result['model']
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

df_comparison = pd.DataFrame(comparison_rows)
csv_path = TABLES_DIR / 'val_comparison_metrics.csv'
json_path = TABLES_DIR / 'val_comparison_metrics.json'
df_comparison.to_csv(csv_path, index=False)
with open(json_path, 'w') as f:
    json.dump(comparison_rows, f, indent=2, ensure_ascii=False)
print("Confronto salvato in:", csv_path)
display(df_comparison)


In [ ]:
metric_keys = ['roc_auc', 'accuracy', 'precision_malato', 'recall_malato', 'f1_malato']
metric_labels = ['AUC', 'Accuracy', 'Precisione\nmalato', 'Recall\nmalato', 'F1\nmalato']

x = np.arange(len(metric_keys))
width = 0.8 / len(comparison_rows)
fig, ax = plt.subplots(figsize=(11, 5))
for i, row in enumerate(comparison_rows):
    vals = [row[k] for k in metric_keys]
    bars = ax.bar(x + i * width, vals, width, label=row['name'])
    for bar, value in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, value + 0.005,
                f"{value:.3f}", ha='center', va='bottom', fontsize=8)

ax.set_xticks(x + width * (len(comparison_rows) - 1) / 2)
ax.set_xticklabels(metric_labels)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Score')
ax.set_title('Synthetic-only from-scratch SD-VAE - partial vs full')
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.5)
fig.tight_layout()
plot_path = FIGURES_DIR / 'val_metrics_partial_vs_full.png'
fig.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print("Grafico salvato in:", plot_path)

fig, ax = plt.subplots(figsize=(8, 6))
for name, out in variant_outputs.items():
    fpr, tpr, _ = roc_curve(out['y_true'], out['y_prob'])
    auc = roc_auc_score(out['y_true'], out['y_prob'])
    ax.plot(fpr, tpr, lw=2, label=f"{name} (AUC={auc:.4f})")
ax.plot([0, 1], [0, 1], '--', color='gray')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC validation - synthetic-only from-scratch SD-VAE')
ax.legend(loc='lower right')
ax.grid(True, linestyle='--', alpha=0.5)
fig.tight_layout()
roc_path = FIGURES_DIR / 'val_roc_partial_vs_full.png'
fig.savefig(roc_path, dpi=150, bbox_inches='tight')
plt.show()
print("ROC salvata in:", roc_path)


In [ ]:
pred_df = df_val[['processed_path', 'cancer']].copy().rename(columns={'cancer': 'label_true'})
for name, out in variant_outputs.items():
    safe = name.lower().replace('+', '_').replace(' ', '_')
    pred_df[f'prob_{safe}'] = out['y_prob']
    pred_df[f'pred_{safe}'] = (out['y_prob'] >= out['threshold']).astype(int)
pred_path = PREDICTIONS_DIR / 'val_predictions_partial_vs_full.csv'
pred_df.to_csv(pred_path, index=False)
print("Predizioni salvate in:", pred_path)

selection_df = df_comparison.copy()
selection_df = selection_df.sort_values(['roc_auc', 'f1_malato', 'recall_malato'], ascending=False)
recommended = selection_df.iloc[0].to_dict()
with open(TABLES_DIR / 'recommended_fine_tune_mode.json', 'w') as f:
    json.dump(recommended, f, indent=2, ensure_ascii=False)
print("Modalita consigliata su validation:")
print(json.dumps(recommended, indent=2, ensure_ascii=False))
